# GBIF Data Transformation

## Forest Explorer Kenya

### Objective

This notebook cleans and transforms the raw GBIF occurrence data into a structured dataset suitable for analysis, visualization, and application development.

### Transformation Goals

- Inspect data quality
- Remove duplicate records
- Handle missing values
- Remove unnecessary columns
- Rename important columns
- Standardize text fields
- Convert date columns
- Validate geographic coordinates
- Export the cleaned dataset

In [2]:
import json
import pandas as pd
import pyarrow

In [3]:
with open("../data/raw/gbif/gbif_test.json", "r", encoding="utf-8") as file:
    data = json.load(file)

df = pd.json_normalize(data["results"])

## Initial Dataset Overview

Before cleaning, we inspect the size, structure, data types, and missing values to understand the quality of the raw dataset.

In [4]:
df.shape

(10, 146)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Columns: 146 entries, key to classifications.d7dddbf4-2cf0-4f39-9b2a-bb099caae36c.acceptedUsage.infraspecificEpithet
dtypes: bool(2), float64(4), int64(15), object(14), str(111)
memory usage: 27.2+ KB


In [5]:
df.isnull().sum()

key                                                                                        0
datasetKey                                                                                 0
publishingOrgKey                                                                           0
installationKey                                                                            0
hostingOrganizationKey                                                                     0
                                                                                          ..
infraspecificEpithet                                                                       9
classifications.7ddf754f-d193-4cc9-b351-99906754a03b.usage.infraspecificEpithet            9
classifications.7ddf754f-d193-4cc9-b351-99906754a03b.acceptedUsage.infraspecificEpithet    9
classifications.d7dddbf4-2cf0-4f39-9b2a-bb099caae36c.usage.infraspecificEpithet            9
classifications.d7dddbf4-2cf0-4f39-9b2a-bb099caae36c.acceptedUsage.inf

# Step 1: Remove Duplicate Records

Duplicate observations can bias analyses and lead to inaccurate species counts. We check for duplicates and remove them if present.

In [6]:
df.duplicated().sum()

TypeError: unhashable type: 'list'

In [7]:
for col in df.columns:
    if df[col].apply(lambda x: isinstance(x, list)).any():
        print(col)

issues
identifiers
media
facts
relations
dnaSequenceID
nucleotideSequence
recordedByIDs
identifiedByIDs
extensions.http://rs.gbif.org/terms/1.0/Multimedia
classifications.7ddf754f-d193-4cc9-b351-99906754a03b.classification
classifications.7ddf754f-d193-4cc9-b351-99906754a03b.issues
classifications.d7dddbf4-2cf0-4f39-9b2a-bb099caae36c.classification
classifications.d7dddbf4-2cf0-4f39-9b2a-bb099caae36c.issues


In [9]:
for col in df.columns:
    if df[col].apply(lambda x: isinstance(x, dict)).any():
        print(col)

## Step 1: Remove Nested Metadata Columns

The raw GBIF dataset contains several nested list and metadata fields generated by the GBIF API. These columns store multimedia references, identifiers, DNA information, classification metadata, and other auxiliary information.

Since the Forest Explorer Kenya project focuses on species distribution, taxonomy, and geographic occurrence data, these fields are not required for analysis.

Removing them also prevents errors when checking for duplicate records because list-type columns cannot be hashed by pandas.

In [10]:
columns_to_drop = [
    "issues",
    "identifiers",
    "media",
    "facts",
    "relations",
    "dnaSequenceID",
    "nucleotideSequence",
    "recordedByIDs",
    "identifiedByIDs",
    "extensions.http://rs.gbif.org/terms/1.0/Multimedia",
    "classifications.7ddf754f-d193-4cc9-b351-99906754a03b.classification",
    "classifications.7ddf754f-d193-4cc9-b351-99906754a03b.issues",
    "classifications.d7dddbf4-2cf0-4f39-9b2a-bb099caae36c.classification",
    "classifications.d7dddbf4-2cf0-4f39-9b2a-bb099caae36c.issues"
]

df = df.drop(columns=columns_to_drop, errors="ignore")

In [11]:
df.duplicated().sum()

np.int64(0)

In [12]:
df = df.drop_duplicates()

## Step 2: Handle Missing Values

Missing values are common in biodiversity datasets because not every observation contains complete information.

Rather than removing all missing values, we first identify which columns are essential for the Forest Explorer Kenya project. Essential columns such as species names and geographic coordinates should not contain missing values, while optional metadata can be retained or removed depending on its usefulness.

In [13]:
# Count Missing Values
# This will show the 20 columns with the most missing values.

missing = df.isnull().sum().sort_values(ascending=False)
missing.head(20)

classifications.d7dddbf4-2cf0-4f39-9b2a-bb099caae36c.acceptedUsage.infraspecificEpithet    9
classifications.d7dddbf4-2cf0-4f39-9b2a-bb099caae36c.usage.infraspecificEpithet            9
classifications.7ddf754f-d193-4cc9-b351-99906754a03b.usage.infraspecificEpithet            9
classifications.7ddf754f-d193-4cc9-b351-99906754a03b.acceptedUsage.infraspecificEpithet    9
projectId                                                                                  9
infraspecificEpithet                                                                       9
dynamicProperties                                                                          8
lifeStage                                                                                  8
vitality                                                                                   8
informationWithheld                                                                        8
classifications.d7dddbf4-2cf0-4f39-9b2a-bb099caae36c.usage.authorship 

In [14]:
# Calculate Missing Percentages
# This tells us how much of each column is missing, which is much more useful than raw counts.

missing_percent = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)

missing_percent.head(20)

classifications.d7dddbf4-2cf0-4f39-9b2a-bb099caae36c.acceptedUsage.infraspecificEpithet    90.0
classifications.d7dddbf4-2cf0-4f39-9b2a-bb099caae36c.usage.infraspecificEpithet            90.0
classifications.7ddf754f-d193-4cc9-b351-99906754a03b.usage.infraspecificEpithet            90.0
classifications.7ddf754f-d193-4cc9-b351-99906754a03b.acceptedUsage.infraspecificEpithet    90.0
projectId                                                                                  90.0
infraspecificEpithet                                                                       90.0
dynamicProperties                                                                          80.0
lifeStage                                                                                  80.0
vitality                                                                                   80.0
informationWithheld                                                                        80.0
classifications.d7dddbf4-2cf0-4f39-9b2a-

In [15]:
# Create a Summary Table
# This produces a clean table that is easier to read and discuss.

missing_summary = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Percentage": round(df.isnull().sum() / len(df) * 100, 2)
})

missing_summary.sort_values("Percentage", ascending=False).head(20)

,Missing Values,Percentage
classifications.d7dddbf4-2cf0-4f39-9b2a-bb099caae36c.acceptedUsage.infraspecificEpithet,9,90.0
classifications.d7dddbf4-2cf0-4f39-9b2a-bb099caae36c.usage.infraspecificEpithet,9,90.0
classifications.7ddf754f-d193-4cc9-b351-99906754a03b.usage.infraspecificEpithet,9,90.0
classifications.7ddf754f-d193-4cc9-b351-99906754a03b.acceptedUsage.infraspecificEpithet,9,90.0
projectId,9,90.0
infraspecificEpithet,9,90.0
dynamicProperties,8,80.0
lifeStage,8,80.0
vitality,8,80.0
informationWithheld,8,80.0


## Step 3: Remove Highly Incomplete Columns

Some columns contain missing values in 80–90% of the records or represent internal GBIF classification metadata.

Since these fields contribute little to the objectives of the Forest Explorer Kenya project, they are removed to simplify the dataset and improve data quality.

In [16]:
# Find columns with more than 70% missing values
high_missing = missing_percent[missing_percent > 70].index.tolist()

high_missing

['classifications.d7dddbf4-2cf0-4f39-9b2a-bb099caae36c.acceptedUsage.infraspecificEpithet',
 'classifications.d7dddbf4-2cf0-4f39-9b2a-bb099caae36c.usage.infraspecificEpithet',
 'classifications.7ddf754f-d193-4cc9-b351-99906754a03b.usage.infraspecificEpithet',
 'classifications.7ddf754f-d193-4cc9-b351-99906754a03b.acceptedUsage.infraspecificEpithet',
 'projectId',
 'infraspecificEpithet',
 'dynamicProperties',
 'lifeStage',
 'vitality',
 'informationWithheld']

In [17]:
# Drop them

df = df.drop(columns=high_missing)

print(f"Remaining columns: {df.shape[1]}")

Remaining columns: 122


In [18]:
# Verification

missing_summary = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Percentage": round(df.isnull().sum() / len(df) * 100, 2)
})

missing_summary.sort_values("Percentage", ascending=False).head(15)

,Missing Values,Percentage
scientificNameAuthorship,1,10.0
orderKey,1,10.0
iucnRedListCategory,1,10.0
coordinateUncertaintyInMeters,1,10.0
order,1,10.0
classifications.d7dddbf4-2cf0-4f39-9b2a-bb099caae36c.acceptedUsage.code,1,10.0
classifications.d7dddbf4-2cf0-4f39-9b2a-bb099caae36c.acceptedUsage.authorship,1,10.0
classifications.d7dddbf4-2cf0-4f39-9b2a-bb099caae36c.iucnRedListCategoryCode,1,10.0
gadm.level2.name,1,10.0
gadm.level3.gid,1,10.0


## Step 4: Rename Columns

To improve readability and maintain consistency across the project, selected columns are renamed using snake_case naming conventions.

Benefits:
- Easier to reference in code
- Consistent naming style
- Better for dashboards and databases
- Improves project maintainability

In [19]:
# Create a dictionary

rename_columns = {
    "scientificName": "scientific_name",
    "country": "country",
    "countryCode": "country_code",
    "stateProvince": "county",
    "verbatimLocality": "locality",
    "decimalLatitude": "latitude",
    "decimalLongitude": "longitude",

    "kingdom": "kingdom",
    "phylum": "phylum",
    "class": "class_name",
    "order": "order_name",
    "family": "family",
    "genus": "genus",
    "species": "species",

    "occurrenceStatus": "occurrence_status",
    "eventDate": "event_date",
    "year": "year",
    "month": "month",
    "day": "day",

    "basisOfRecord": "basis_of_record",
    "institutionCode": "institution_code",
    "collectionCode": "collection_code",
    "catalogNumber": "catalog_number",

    "taxonRank": "taxon_rank",
    "taxonomicStatus": "taxonomic_status",

    "coordinateUncertaintyInMeters": "coordinate_uncertainty_m",
}

In [20]:
# Rename

df.rename(columns=rename_columns, inplace=True)

In [21]:
# Verification

df.columns.tolist()


['key',
 'datasetKey',
 'publishingOrgKey',
 'installationKey',
 'hostingOrganizationKey',
 'publishingCountry',
 'protocol',
 'lastCrawled',
 'lastParsed',
 'crawlId',
 'basis_of_record',
 'occurrence_status',
 'taxonKey',
 'kingdomKey',
 'phylumKey',
 'classKey',
 'orderKey',
 'familyKey',
 'genusKey',
 'speciesKey',
 'acceptedTaxonKey',
 'scientific_name',
 'scientificNameAuthorship',
 'acceptedScientificName',
 'kingdom',
 'phylum',
 'order_name',
 'family',
 'genus',
 'species',
 'genericName',
 'specificEpithet',
 'taxon_rank',
 'taxonomic_status',
 'iucnRedListCategory',
 'dateIdentified',
 'latitude',
 'longitude',
 'coordinate_uncertainty_m',
 'continent',
 'county',
 'year',
 'month',
 'day',
 'event_date',
 'startDayOfYear',
 'endDayOfYear',
 'modified',
 'lastInterpreted',
 'references',
 'license',
 'isSequenced',
 'isInCluster',
 'datasetName',
 'recordedBy',
 'identifiedBy',
 'geodeticDatum',
 'class_name',
 'country_code',
 'gbifRegion',
 'country',
 'publishedByGbifReg

In [22]:
# Reorder most important columns

important_columns = [
    "scientific_name",
    "kingdom",
    "phylum",
    "class_name",
    "order_name",
    "family",
    "genus",
    "species",
    "country",
    "county",
    "locality",
    "latitude",
    "longitude",
    "year",
    "month",
    "day"
]

remaining = [col for col in df.columns if col not in important_columns]

df = df[important_columns + remaining]

In [23]:
df.head()

,scientific_name,kingdom,phylum,class_name,order_name,family,genus,species,country,county,...,classifications.d7dddbf4-2cf0-4f39-9b2a-bb099caae36c.taxonomicStatus,classifications.d7dddbf4-2cf0-4f39-9b2a-bb099caae36c.iucnRedListCategoryCode,gadm.level0.gid,gadm.level0.name,gadm.level1.gid,gadm.level1.name,gadm.level2.gid,gadm.level2.name,gadm.level3.gid,gadm.level3.name
0,"Tursiops aduncus (Ehrenberg, 1833)",Animalia,Chordata,Mammalia,Cetacea,Delphinidae,Tursiops,Tursiops aduncus,Kenya,Kilifi,...,ACCEPTED,NT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"Buteo augur (Ruppell, 1836)",Animalia,Chordata,Aves,Accipitriformes,Accipitridae,Buteo,Buteo augur,Kenya,Narok,...,ACCEPTED,LC,KEN,Kenya,KEN.33_1,Narok,KEN.33.2_1,Kilgoris,KEN.33.2.4_1,Kimintet
2,"Streptopelia capicola (Sundevall, 1857)",Animalia,Chordata,Aves,Columbiformes,Columbidae,Streptopelia,Streptopelia capicola,Kenya,Narok,...,ACCEPTED,LC,KEN,Kenya,KEN.33_1,Narok,KEN.33.6_1,Narok West,KEN.33.6.2_1,Mara
3,"Anastomus lamelligerus Temminck, 1823",Animalia,Chordata,Aves,Ciconiiformes,Ciconiidae,Anastomus,Anastomus lamelligerus,Kenya,Nakuru,...,ACCEPTED,LC,KEN,Kenya,KEN.31_1,Nakuru,KEN.31.6_1,Naivasha,KEN.31.6.7_1,Olkaria
4,"Trioceros jacksonii (Boulenger, 1896)",Animalia,Chordata,Squamata,NaN,Chamaeleonidae,Trioceros,Trioceros jacksonii,Kenya,Nairobi,...,ACCEPTED,LC,KEN,Kenya,KEN.30_1,Nairobi,KEN.30.11_1,Langata,KEN.30.11.1_1,Karen


# Step 5: Standardize and Clean Values

This step ensures that all text, dates and numerical values follow a consistent format. Standardized data is easier to analyze, visualize and load into a database.

### 5.1 Remove Leading and Trailing Spaces

Some text fields contain accidental spaces.

In [24]:
text_columns = df.select_dtypes(include="object").columns

for col in text_columns:
    df[col] = df[col].str.strip()

C:\Users\Owner\AppData\Local\Temp\ipykernel_17012\3659800821.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  text_columns = df.select_dtypes(include="object").columns


In [25]:
# Verification

df.head()

,scientific_name,kingdom,phylum,class_name,order_name,family,genus,species,country,county,...,classifications.d7dddbf4-2cf0-4f39-9b2a-bb099caae36c.taxonomicStatus,classifications.d7dddbf4-2cf0-4f39-9b2a-bb099caae36c.iucnRedListCategoryCode,gadm.level0.gid,gadm.level0.name,gadm.level1.gid,gadm.level1.name,gadm.level2.gid,gadm.level2.name,gadm.level3.gid,gadm.level3.name
0,"Tursiops aduncus (Ehrenberg, 1833)",Animalia,Chordata,Mammalia,Cetacea,Delphinidae,Tursiops,Tursiops aduncus,Kenya,Kilifi,...,ACCEPTED,NT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"Buteo augur (Ruppell, 1836)",Animalia,Chordata,Aves,Accipitriformes,Accipitridae,Buteo,Buteo augur,Kenya,Narok,...,ACCEPTED,LC,KEN,Kenya,KEN.33_1,Narok,KEN.33.2_1,Kilgoris,KEN.33.2.4_1,Kimintet
2,"Streptopelia capicola (Sundevall, 1857)",Animalia,Chordata,Aves,Columbiformes,Columbidae,Streptopelia,Streptopelia capicola,Kenya,Narok,...,ACCEPTED,LC,KEN,Kenya,KEN.33_1,Narok,KEN.33.6_1,Narok West,KEN.33.6.2_1,Mara
3,"Anastomus lamelligerus Temminck, 1823",Animalia,Chordata,Aves,Ciconiiformes,Ciconiidae,Anastomus,Anastomus lamelligerus,Kenya,Nakuru,...,ACCEPTED,LC,KEN,Kenya,KEN.31_1,Nakuru,KEN.31.6_1,Naivasha,KEN.31.6.7_1,Olkaria
4,"Trioceros jacksonii (Boulenger, 1896)",Animalia,Chordata,Squamata,NaN,Chamaeleonidae,Trioceros,Trioceros jacksonii,Kenya,Nairobi,...,ACCEPTED,LC,KEN,Kenya,KEN.30_1,Nairobi,KEN.30.11_1,Langata,KEN.30.11.1_1,Karen


### 5.2 Standardize Country Names

Checking unique country values.

In [26]:
df["country"].unique()

<StringArray>
['Kenya']
Length: 1, dtype: str

In [27]:
# Standardization

df["country"] = df["country"].str.title()

### 5.3 Standardize County Names

Check them.

In [28]:
sorted(df["county"].dropna().unique())

['Kilifi', 'Kwale', 'Nairobi', 'Nakuru', 'Narok', 'Taita Taveta']

### 5.4 Standardize Scientific Names

Scientific names should preserve their scientific capitalization.

In [29]:
# Removing spaces.

df["scientific_name"] = df["scientific_name"].str.strip()

### 5.5 Standardize Country Codes

Check them.

In [30]:
df["country_code"].unique()

<StringArray>
['KE']
Length: 1, dtype: str

### 5.6 Remove Double Spaces

Sometimes locality names contain extra spaces.

In [31]:
for col in ["scientific_name",
            "country",
            "county",
            "locality"]:
    df[col] = df[col].str.replace(r"\s+", " ", regex=True)

### 5.7 Check Coordinates

Latitude must be between

-90 and 90

Longitude

-180 and 180

In [32]:
df[
    (df["latitude"] < -90) |
    (df["latitude"] > 90)
]

,scientific_name,kingdom,phylum,class_name,order_name,family,genus,species,country,county,...,classifications.d7dddbf4-2cf0-4f39-9b2a-bb099caae36c.taxonomicStatus,classifications.d7dddbf4-2cf0-4f39-9b2a-bb099caae36c.iucnRedListCategoryCode,gadm.level0.gid,gadm.level0.name,gadm.level1.gid,gadm.level1.name,gadm.level2.gid,gadm.level2.name,gadm.level3.gid,gadm.level3.name


In [33]:
df[
    (df["longitude"] < -180) |
    (df["longitude"] > 180)
]

,scientific_name,kingdom,phylum,class_name,order_name,family,genus,species,country,county,...,classifications.d7dddbf4-2cf0-4f39-9b2a-bb099caae36c.taxonomicStatus,classifications.d7dddbf4-2cf0-4f39-9b2a-bb099caae36c.iucnRedListCategoryCode,gadm.level0.gid,gadm.level0.name,gadm.level1.gid,gadm.level1.name,gadm.level2.gid,gadm.level2.name,gadm.level3.gid,gadm.level3.name


### 5.8 Ensure Numeric Data Types

In [34]:
df["latitude"] = pd.to_numeric(df["latitude"])
df["longitude"] = pd.to_numeric(df["longitude"])

In [35]:
df.dtypes

scientific_name     str
kingdom             str
phylum              str
class_name          str
order_name          str
                   ... 
gadm.level1.name    str
gadm.level2.gid     str
gadm.level2.name    str
gadm.level3.gid     str
gadm.level3.name    str
Length: 122, dtype: object

### 5.9 Final Quality Check



In [36]:
# Check unique countries

df["country"].value_counts()

country
Kenya    10
Name: count, dtype: int64

In [37]:
# Check county names

df["county"].value_counts().head(20)

county
Narok           5
Kilifi          1
Nakuru          1
Nairobi         1
Taita Taveta    1
Kwale           1
Name: count, dtype: int64

In [38]:
# Check country code

df["country_code"].value_counts()

country_code
KE    10
Name: count, dtype: int64

### 5.10 Preview the Cleaned Dataset

In [39]:
df.head()

,scientific_name,kingdom,phylum,class_name,order_name,family,genus,species,country,county,...,classifications.d7dddbf4-2cf0-4f39-9b2a-bb099caae36c.taxonomicStatus,classifications.d7dddbf4-2cf0-4f39-9b2a-bb099caae36c.iucnRedListCategoryCode,gadm.level0.gid,gadm.level0.name,gadm.level1.gid,gadm.level1.name,gadm.level2.gid,gadm.level2.name,gadm.level3.gid,gadm.level3.name
0,"Tursiops aduncus (Ehrenberg, 1833)",Animalia,Chordata,Mammalia,Cetacea,Delphinidae,Tursiops,Tursiops aduncus,Kenya,Kilifi,...,ACCEPTED,NT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,"Buteo augur (Ruppell, 1836)",Animalia,Chordata,Aves,Accipitriformes,Accipitridae,Buteo,Buteo augur,Kenya,Narok,...,ACCEPTED,LC,KEN,Kenya,KEN.33_1,Narok,KEN.33.2_1,Kilgoris,KEN.33.2.4_1,Kimintet
2,"Streptopelia capicola (Sundevall, 1857)",Animalia,Chordata,Aves,Columbiformes,Columbidae,Streptopelia,Streptopelia capicola,Kenya,Narok,...,ACCEPTED,LC,KEN,Kenya,KEN.33_1,Narok,KEN.33.6_1,Narok West,KEN.33.6.2_1,Mara
3,"Anastomus lamelligerus Temminck, 1823",Animalia,Chordata,Aves,Ciconiiformes,Ciconiidae,Anastomus,Anastomus lamelligerus,Kenya,Nakuru,...,ACCEPTED,LC,KEN,Kenya,KEN.31_1,Nakuru,KEN.31.6_1,Naivasha,KEN.31.6.7_1,Olkaria
4,"Trioceros jacksonii (Boulenger, 1896)",Animalia,Chordata,Squamata,NaN,Chamaeleonidae,Trioceros,Trioceros jacksonii,Kenya,Nairobi,...,ACCEPTED,LC,KEN,Kenya,KEN.30_1,Nairobi,KEN.30.11_1,Langata,KEN.30.11.1_1,Karen


# Step 6: Feature Engineering

Creating new variables (features) from existing data to improve analysis and make the dataset more useful for visualization, querying and future applications.

The new features created will support:
- Species distribution analysis
- Geographic visualization
- Time-based analysis
- Data quality assessment
- Search and filtering in the Forest Explorer Kenya application

### 6.1 Creating a Full Location Column

In [40]:
# Instead of having county and country separately, create one readable location.

df["location"] = (
    df["county"].fillna("Unknown")
    + ", "
    + df["country"].fillna("Unknown")
)

In [41]:
df[["county","country","location"]].head()

,county,country,location
0,Kilifi,Kenya,"Kilifi, Kenya"
1,Narok,Kenya,"Narok, Kenya"
2,Narok,Kenya,"Narok, Kenya"
3,Nakuru,Kenya,"Nakuru, Kenya"
4,Nairobi,Kenya,"Nairobi, Kenya"


### 6.2 Creating Coordinate Availability

In [42]:
# Let's create a simple Yes/No column.

df["has_coordinates"] = (
    df["latitude"].notna()
    & df["longitude"].notna()
)

In [43]:
df["has_coordinates"].value_counts()

has_coordinates
True    10
Name: count, dtype: int64

### 6.3 Observation Year

In [44]:
# Creating a cleaner year column.

if "year" in df.columns:
    df["observation_year"] = df["year"]

In [45]:
df["observation_year"].value_counts().sort_index()

observation_year
2026    10
Name: count, dtype: int64

### 6.4 Month Name

In [46]:
# Checking if month exists

import calendar

if "month" in df.columns:
    df["observation_month"] = df["month"].apply(
        lambda x: calendar.month_name[int(x)]
        if pd.notna(x) else None
    )

In [47]:
df[["month","observation_month"]].head()

,month,observation_month
0,1,January
1,1,January
2,1,January
3,1,January
4,1,January


### 6.5 Species Name Length

- Useful for text analysis.

In [48]:
df["scientific_name_length"] = df["scientific_name"].str.len()

In [49]:
df[
    ["scientific_name",
     "scientific_name_length"]
].head()

,scientific_name,scientific_name_length
0,"Tursiops aduncus (Ehrenberg, 1833)",34
1,"Buteo augur (Ruppell, 1836)",27
2,"Streptopelia capicola (Sundevall, 1857)",39
3,"Anastomus lamelligerus Temminck, 1823",37
4,"Trioceros jacksonii (Boulenger, 1896)",37


### 6.6 Number of Words in Scientific Name

- Many species names are binomial.

In [50]:
df["scientific_name_words"] = (
    df["scientific_name"]
    .str.split()
    .str.len()
)

In [51]:
df[
    ["scientific_name",
     "scientific_name_words"]
].head()

,scientific_name,scientific_name_words
0,"Tursiops aduncus (Ehrenberg, 1833)",4
1,"Buteo augur (Ruppell, 1836)",4
2,"Streptopelia capicola (Sundevall, 1857)",4
3,"Anastomus lamelligerus Temminck, 1823",4
4,"Trioceros jacksonii (Boulenger, 1896)",4


### 6.7 Creating Species Initial

In [52]:
df["species_initial"] = (
    df["scientific_name"]
    .str[0]
    .str.upper()
)

In [53]:
df[
    ["scientific_name",
     "species_initial"]
].head()

,scientific_name,species_initial
0,"Tursiops aduncus (Ehrenberg, 1833)",T
1,"Buteo augur (Ruppell, 1836)",B
2,"Streptopelia capicola (Sundevall, 1857)",S
3,"Anastomus lamelligerus Temminck, 1823",A
4,"Trioceros jacksonii (Boulenger, 1896)",T


### 6.8 Coordinate Precision

In [54]:
# If coordinate uncertainty exists

if "coordinate_uncertainty_m" in df.columns:
    df["high_precision"] = (
        df["coordinate_uncertainty_m"] < 100
    )

In [55]:
# Tells us whether an observation has relatively precise coordinates.

df[
    ["coordinate_uncertainty_m",
     "high_precision"]
].head()

,coordinate_uncertainty_m,high_precision
0,2.0,True
1,NaN,False
2,11853.0,False
3,34.0,True
4,64.0,True


### 6.9 Conservation Status Availability

- Some species records include an IUCN Red List conservation status. This feature identifies whether a conservation status is available for each observation.

In [56]:
df["has_iucn_status"] = df["iucnRedListCategory"].notna()

In [57]:
df["has_iucn_status"].value_counts()

has_iucn_status
True     9
False    1
Name: count, dtype: int64

### 6.10 Dataset Completeness Score 

- This is an excellent feature for assessing data quality.

In [58]:
important_columns = [
    "scientific_name",
    "country",
    "county",
    "latitude",
    "longitude"
]

df["completeness_score"] = (
    df[important_columns]
    .notna()
    .sum(axis=1)
)

Score meanings:

5 = Complete record

4 = Good record

3 = Fair

2 = Poor

1 = Very Poor

In [59]:
df[
    important_columns +
    ["completeness_score"]
].head()

,scientific_name,country,county,latitude,longitude,completeness_score
0,"Tursiops aduncus (Ehrenberg, 1833)",Kenya,Kilifi,-3.430600,39.964508,5
1,"Buteo augur (Ruppell, 1836)",Kenya,Narok,-1.272472,34.985667,5
2,"Streptopelia capicola (Sundevall, 1857)",Kenya,Narok,-1.124248,35.267638,5
3,"Anastomus lamelligerus Temminck, 1823",Kenya,Nakuru,-0.826978,36.374481,5
4,"Trioceros jacksonii (Boulenger, 1896)",Kenya,Nairobi,-1.379105,36.765167,5


### 6.11 Preview the New Features

In [60]:
new_features = [
    "location",
    "has_coordinates",
    "observation_year",
    "observation_month",
    "scientific_name_length",
    "scientific_name_words",
    "species_initial",
    "completeness_score"
]

existing_features = [col for col in new_features if col in df.columns]

df[existing_features].head()

,location,has_coordinates,observation_year,observation_month,scientific_name_length,scientific_name_words,species_initial,completeness_score
0,"Kilifi, Kenya",True,2026,January,34,4,T,5
1,"Narok, Kenya",True,2026,January,27,4,B,5
2,"Narok, Kenya",True,2026,January,39,4,S,5
3,"Nakuru, Kenya",True,2026,January,37,4,A,5
4,"Nairobi, Kenya",True,2026,January,37,4,T,5


### 6.12 Create a Taxonomy Hierarchy

A taxonomy hierarchy combines the biological classification of each species into a single readable string. This makes it easier to display taxonomic information.

In [61]:
df["taxonomy"] = (
    df["kingdom"].fillna("Unknown") + " > " +
    df["phylum"].fillna("Unknown") + " > " +
    df["class_name"].fillna("Unknown") + " > " +
    df["order_name"].fillna("Unknown") + " > " +
    df["family"].fillna("Unknown") + " > " +
    df["genus"].fillna("Unknown") + " > " +
    df["species"].fillna("Unknown")
)

In [62]:
df[["scientific_name", "taxonomy"]].head()

,scientific_name,taxonomy
0,"Tursiops aduncus (Ehrenberg, 1833)",Animalia > Chordata > Mammalia > Cetacea > Del...
1,"Buteo augur (Ruppell, 1836)",Animalia > Chordata > Aves > Accipitriformes >...
2,"Streptopelia capicola (Sundevall, 1857)",Animalia > Chordata > Aves > Columbiformes > C...
3,"Anastomus lamelligerus Temminck, 1823",Animalia > Chordata > Aves > Ciconiiformes > C...
4,"Trioceros jacksonii (Boulenger, 1896)",Animalia > Chordata > Squamata > Unknown > Cha...


### 6.13 Create Google Maps Links

Each observation is assigned a Google Maps URL based on its latitude and longitude, allowing users to open the observation location directly in Google Maps.

In [63]:
df["google_maps_url"] = (
    "https://www.google.com/maps?q="
    + df["latitude"].astype(str)
    + ","
    + df["longitude"].astype(str)
)

In [64]:
df[["latitude", "longitude", "google_maps_url"]].head()

,latitude,longitude,google_maps_url
0,-3.430600,39.964508,"https://www.google.com/maps?q=-3.4306,39.964508"
1,-1.272472,34.985667,"https://www.google.com/maps?q=-1.272472,34.985667"
2,-1.124248,35.267638,"https://www.google.com/maps?q=-1.124248,35.267638"
3,-0.826978,36.374481,"https://www.google.com/maps?q=-0.826978,36.374481"
4,-1.379105,36.765167,"https://www.google.com/maps?q=-1.379105,36.765167"


### 6.14 Classify Observation Season

Each observation is assigned to one of Kenya's four climatic seasons based on the observation month.

In [65]:
def season(month):

    if pd.isna(month):
        return "Unknown"

    month = int(month)

    if month in [12, 1, 2]:
        return "Dry Season"

    elif month in [3, 4, 5]:
        return "Long Rains"

    elif month in [6, 7, 8, 9]:
        return "Cool Dry"

    else:
        return "Short Rains"

df["season"] = df["month"].apply(season)

In [66]:
df[["month", "season"]].head()

,month,season
0,1,Dry Season
1,1,Dry Season
2,1,Dry Season
3,1,Dry Season
4,1,Dry Season


### 6.15 Validate Geographic Coordinates

This feature checks whether observation coordinates fall within the approximate geographic boundaries of Kenya.

In [67]:
df["valid_coordinates"] = (
    df["latitude"].between(-5, 5) &
    df["longitude"].between(33, 42)
)

In [68]:
df["valid_coordinates"].value_counts()

valid_coordinates
True    10
Name: count, dtype: int64

### 6.16 Create Full Location

A readable location string is created by combining locality, county and country.

In [69]:
df["full_location"] = (
    df["locality"].fillna("Unknown") + ", " +
    df["county"].fillna("Unknown") + ", " +
    df["country"].fillna("Unknown")
)

In [70]:
df[["locality", "county", "full_location"]].head()

,locality,county,full_location
0,"Kilifi, KE",Kilifi,"Kilifi, KE, Kilifi, Kenya"
1,"Lolgorian, Kenya",Narok,"Lolgorian, Kenya, Narok, Kenya"
2,"Ololunga, Kenya",Narok,"Ololunga, Kenya, Narok, Kenya"
3,"Sulmac Village, Kenya",Nakuru,"Sulmac Village, Kenya, Nakuru, Kenya"
4,"Nairobi, Nairobi, KE",Nairobi,"Nairobi, Nairobi, KE, Nairobi, Kenya"


### 6.17 Calculate Observation Age

The number of days since each observation was recorded is calculated using the event date.

In [71]:
df["event_date"] = pd.to_datetime(df["event_date"], errors="coerce")

df["days_since_observation"] = (
    pd.Timestamp.today() - df["event_date"]
).dt.days

In [72]:
df[["event_date", "days_since_observation"]].head()

,event_date,days_since_observation
0,2026-01-02 09:01:39,199.0
1,NaT,NaN
2,NaT,NaN
3,NaT,NaN
4,2026-01-04 20:37:29,196.0


### 6.18 Create Wildlife Groups

Species are grouped into broader wildlife categories based on their biological class, making filtering and visualization easier for users.

In [73]:
def wildlife_group(class_name):

    if pd.isna(class_name):
        return "Unknown"

    groups = {
        "Mammalia": "Mammal",
        "Aves": "Bird",
        "Reptilia": "Reptile",
        "Amphibia": "Amphibian",
        "Insecta": "Insect",
        "Actinopterygii": "Fish"
    }

    return groups.get(class_name, "Other")

df["wildlife_group"] = df["class_name"].apply(wildlife_group)

In [74]:
df["wildlife_group"].value_counts()

wildlife_group
Bird      5
Mammal    4
Other     1
Name: count, dtype: int64

## Step 7: Export Processed Dataset

After cleaning and feature engineering, the processed dataset is exported for downstream analysis, loading into BigQuery and use within the Forest Explorer Kenya application.

The dataset is exported in CSV format for readability and interoperability.

In [9]:
import os

os.makedirs("../data/processed", exist_ok=True)

In [10]:
# Save it
df.to_csv(
    "../data/processed/gbif_clean.csv",
    index=False
)

In [11]:
# Verifying it is saved
import os

os.listdir("../data/processed")

['gbif_clean.csv', 'gbif_clean.parquet']

Save as Parquet - this is the format commonly used in modern data engineering.

In [13]:
df.to_parquet(
    "../data/processed/gbif_clean.parquet",
    index=False
)

In [85]:
os.listdir("../data/processed")

['gbif_clean.csv', 'gbif_clean.parquet']

In [86]:
# Test reading the saved file
df_check = pd.read_csv("../data/processed/gbif_clean.csv")

In [87]:
df_check.shape

(10, 139)

In [88]:
df.shape

(10, 139)

In [89]:
df_check.head()

,scientific_name,kingdom,phylum,class_name,order_name,family,genus,species,country,county,...,high_precision,has_iucn_status,completeness_score,taxonomy,google_maps_url,season,valid_coordinates,full_location,days_since_observation,wildlife_group
0,"Tursiops aduncus (Ehrenberg, 1833)",Animalia,Chordata,Mammalia,Cetacea,Delphinidae,Tursiops,Tursiops aduncus,Kenya,Kilifi,...,True,True,5,Animalia > Chordata > Mammalia > Cetacea > Del...,"https://www.google.com/maps?q=-3.4306,39.964508",Dry Season,True,"Kilifi, KE, Kilifi, Kenya",199.0,Mammal
1,"Buteo augur (Ruppell, 1836)",Animalia,Chordata,Aves,Accipitriformes,Accipitridae,Buteo,Buteo augur,Kenya,Narok,...,False,True,5,Animalia > Chordata > Aves > Accipitriformes >...,"https://www.google.com/maps?q=-1.272472,34.985667",Dry Season,True,"Lolgorian, Kenya, Narok, Kenya",NaN,Bird
2,"Streptopelia capicola (Sundevall, 1857)",Animalia,Chordata,Aves,Columbiformes,Columbidae,Streptopelia,Streptopelia capicola,Kenya,Narok,...,False,True,5,Animalia > Chordata > Aves > Columbiformes > C...,"https://www.google.com/maps?q=-1.124248,35.267638",Dry Season,True,"Ololunga, Kenya, Narok, Kenya",NaN,Bird
3,"Anastomus lamelligerus Temminck, 1823",Animalia,Chordata,Aves,Ciconiiformes,Ciconiidae,Anastomus,Anastomus lamelligerus,Kenya,Nakuru,...,True,True,5,Animalia > Chordata > Aves > Ciconiiformes > C...,"https://www.google.com/maps?q=-0.826978,36.374481",Dry Season,True,"Sulmac Village, Kenya, Nakuru, Kenya",NaN,Bird
4,"Trioceros jacksonii (Boulenger, 1896)",Animalia,Chordata,Squamata,NaN,Chamaeleonidae,Trioceros,Trioceros jacksonii,Kenya,Nairobi,...,True,True,5,Animalia > Chordata > Squamata > Unknown > Cha...,"https://www.google.com/maps?q=-1.379105,36.765167",Dry Season,True,"Nairobi, Nairobi, KE, Nairobi, Kenya",196.0,Other


In [90]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nMissing Values:")
print(df.isnull().sum().sort_values(ascending=False).head(10))

print("\nDuplicate Rows:")
print(df.duplicated().sum())

Rows: 10
Columns: 139

Missing Values:
event_date                  8
days_since_observation      8
order_name                  1
orderKey                    1
gadm.level1.gid             1
gadm.level1.name            1
coordinate_uncertainty_m    1
scientificNameAuthorship    1
iucnRedListCategory         1
gadm.level3.gid             1
dtype: int64

Duplicate Rows:
0


In [ ]:
# BigQuery doesn't allow some special characters in column names
# Checking for the bad columns

print(df.columns.tolist())

['key', 'datasetKey', 'publishingOrgKey', 'installationKey', 'hostingOrganizationKey', 'publishingCountry', 'protocol', 'lastCrawled', 'lastParsed', 'crawlId', 'basisOfRecord', 'occurrenceStatus', 'taxonKey', 'kingdomKey', 'phylumKey', 'classKey', 'orderKey', 'familyKey', 'genusKey', 'speciesKey', 'acceptedTaxonKey', 'scientificName', 'scientificNameAuthorship', 'acceptedScientificName', 'kingdom', 'phylum', 'order', 'family', 'genus', 'species', 'genericName', 'specificEpithet', 'taxonRank', 'taxonomicStatus', 'iucnRedListCategory', 'dateIdentified', 'decimalLatitude', 'decimalLongitude', 'coordinateUncertaintyInMeters', 'continent', 'stateProvince', 'year', 'month', 'day', 'eventDate', 'startDayOfYear', 'endDayOfYear', 'issues', 'modified', 'lastInterpreted', 'references', 'license', 'isSequenced', 'identifiers', 'media', 'facts', 'relations', 'isInCluster', 'datasetName', 'recordedBy', 'identifiedBy', 'dnaSequenceID', 'nucleotideSequence', 'geodeticDatum', 'class', 'countryCode', 'r

In [7]:
# Clean all the column names

import re

df.columns = [
    re.sub(r'[^a-zA-Z0-9_]', '_', col)
    for col in df.columns
]